In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window

silver_df = spark.read.table('fraud_mlops.silver.transactions_clean')
silver_df.printSchema()

In [0]:


gold_base_df = silver_df.withColumn(
    'transaction_id',
    sha2(
        concat_ws(
            '||',
            col('step').cast('string'),
            col('transaction_type'),
            col('amount').cast('string'),
            col('origin_account'),
            col('destination_account')
        ),256
    )    
)


In [0]:
gold_base_df = (
    gold_base_df

    .withColumn(
        "hour_of_day",
        ((col("step") - 1) % 24).cast("integer")
    )

    .withColumn(
        "day_number",
        (floor((col("step") - 1) / 24) + 1).cast("integer")
    )
)

display(gold_base_df.limit(10))

In [0]:
gold_base_df = gold_base_df.withColumn('log_amount',log1p(col('amount')))
display(gold_base_df.limit(10))


### one-hot encoding

In [0]:
gold_base_df = gold_base_df.withColumn('type_cash_in',when(col('transaction_type')=='CASH_IN',1).otherwise(0))\
    .withColumn('type_cash_out',when(col('transaction_type')=='CASH_OUT',1).otherwise(0))\
        .withColumn('type_debit',when(col('transaction_type')=='DEBIT',1).otherwise(0))\
            .withColumn('type_payment',when(col('transaction_type')=='PAYMENT',1).otherwise(0))\
                .withColumn('type_transfer',when(col('transaction_type')=='TRANSFER',1).otherwise(0))


### historical behavior features

In [0]:
origin_history_window = Window.partitionBy(col('origin_account')).orderBy(col('step')).rangeBetween(Window.unboundedPreceding, -1)

In [0]:
gold_base_df = gold_base_df.withColumn('origin_txn_count_prior',count('*').over(origin_history_window))\
    .withColumn('origin_avg_amount_prior',avg(col('amount')).over(origin_history_window))\
        .withColumn('origin_total_amount_prior',sum(col('amount')).over(origin_history_window))



In [0]:
gold_base_df = gold_base_df.withColumn('amount_vs_origin_avg',when(col('origin_avg_amount_prior')>0,col('amount')/col('origin_avg_amount_prior')).otherwise(0.0))


In [0]:
gold_base_df = gold_base_df.withColumn('origin_has_history',when(col('origin_txn_count_prior')>0,1).otherwise(0))\
    .fillna({
        'origin_txn_count_prior':0,
        'origin_avg_amount_prior':0.0,
        'origin_total_amount_prior':0.0,
        'amount_vs_origin_avg':0.0
    })

display(gold_base_df.limit(10))

In [0]:
gold_df = gold_base_df.select('transaction_id',\
    'step','hour_of_day','day_number',\
    'amount','log_amount',\
    'type_cash_in','type_cash_out','type_debit','type_payment','type_transfer',\
    'origin_txn_count_prior','origin_avg_amount_prior','origin_total_amount_prior','amount_vs_origin_avg','origin_has_history',\
    'is_fraud','label_status')

display(gold_df.limit(10))

In [0]:
gold_df.printSchema()

In [0]:
display(gold_df.count())

In [0]:
gold_df=gold_df.withColumn('dataset_split',when(col('step')<=520,'train')\
    .when(col('step')<=632,'validation')\
        .otherwise('test'))



In [0]:
display(gold_df.groupBy('is_fraud','dataset_split').count().orderBy('dataset_split','is_fraud'))

In [0]:
gold_df.write.mode('overwrite').format('delta').option('overwriteSchema','true').saveAsTable('fraud_mlops.gold.transaction_features')

In [0]:
training_snapshot_df =gold_df.filter(col('label_status')=='AVAILABLE')\
    .drop(col('label_status'))

In [0]:
training_snapshot_df.write.mode('overwrite').format('delta').option('overwriteSchema',True).saveAsTable('fraud_mlops.gold.fraud_training_snapshot')

In [0]:
training_df = spark.read.table('fraud_mlops.gold.fraud_training_snapshot')

display(training_df.limit(10))
training_df.printSchema()
print(f'Training rows: {training_df.count()}')

In [0]:
training_df.groupBy('dataset_split').count().show()

In [0]:
display(training_df.groupBy('is_fraud','dataset_split').count().orderBy('dataset_split','is_fraud'))


In [0]:
training_df = spark.table(
    "fraud_mlops.gold.fraud_training_snapshot"
)

print("Rows:", training_df.count())
training_df.printSchema()